In [1]:
import pandas as pd
import os
import re
import numpy as np
import gc
from collections import Counter # Thêm cái này để đếm cho nhanh
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import torch

# --- CẤU HÌNH ---
INPUT_FOLDER = "Clean data"
OUTPUT_FOLDER = "ALIGN"
MODEL_NAME = 'sentence-transformers/LaBSE'
SKIP_PENALTY = -0.6 
MIN_THRESHOLD = 0.4  

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Đang chạy trên thiết bị: {device}")

print("Đang tải model LaBSE...")
model = SentenceTransformer(MODEL_NAME, device=device)

def get_embeddings(sentences):
    if not sentences: return np.array([])
    return model.encode(sentences, show_progress_bar=False, convert_to_numpy=True)

def dp_alignment_with_score(src_sents, tgt_sents, src_emb, tgt_emb):
    n = len(src_sents)
    m = len(tgt_sents)
    
    dp = np.full((n + 1, m + 1), -np.inf)
    dp[0][0] = 0
    trace = np.zeros((n + 1, m + 1), dtype=np.int8)
    
    sim_matrix = cosine_similarity(src_emb, tgt_emb)
    
    for i in range(n + 1):
        for j in range(m + 1):
            if i == 0 and j == 0: continue
            
            # 1. Skip Source
            if i > 0:
                score = dp[i-1][j] + SKIP_PENALTY
                if score > dp[i][j]: dp[i][j] = score; trace[i][j] = 4
            
            # 2. Skip Target
            if j > 0:
                score = dp[i][j-1] + SKIP_PENALTY
                if score > dp[i][j]: dp[i][j] = score; trace[i][j] = 5
            
            # 3. Match 1-1
            if i > 0 and j > 0:
                score = dp[i-1][j-1] + sim_matrix[i-1][j-1]
                if score > dp[i][j]: dp[i][j] = score; trace[i][j] = 1
            
            # 4. Match 1-2 (SPAN TARGET)
            if i > 0 and j > 1:
                avg_tgt_vec = (tgt_emb[j-1] + tgt_emb[j-2]) / 2
                sim = np.dot(src_emb[i-1], avg_tgt_vec) / (np.linalg.norm(src_emb[i-1]) * np.linalg.norm(avg_tgt_vec))
                score = dp[i-1][j-2] + sim
                if score > dp[i][j]: dp[i][j] = score; trace[i][j] = 2
                    
            # 5. Match 2-1 (SPAN SOURCE)
            if i > 1 and j > 0:
                avg_src_vec = (src_emb[i-1] + src_emb[i-2]) / 2
                sim = np.dot(avg_src_vec, tgt_emb[j-1]) / (np.linalg.norm(avg_src_vec) * np.linalg.norm(tgt_emb[j-1]))
                score = dp[i-2][j-1] + sim
                if score > dp[i][j]: dp[i][j] = score; trace[i][j] = 3

    # Truy vết kết quả
    results = []
    curr_i, curr_j = n, m
    while curr_i > 0 or curr_j > 0:
        step = trace[curr_i][curr_j]
        
        # Thêm tham số "Type" vào kết quả trả về để thống kê
        if step == 1: # 1-1
            score = float(sim_matrix[curr_i-1][curr_j-1])
            results.append((src_sents[curr_i-1], tgt_sents[curr_j-1], score, "1_1"))
            curr_i -= 1; curr_j -= 1
            
        elif step == 2: # 1-2
            combined_tgt = tgt_sents[curr_j-2] + " " + tgt_sents[curr_j-1]
            avg_tgt_vec = (tgt_emb[curr_j-1] + tgt_emb[curr_j-2]) / 2
            score = float(np.dot(src_emb[curr_i-1], avg_tgt_vec) / (np.linalg.norm(src_emb[curr_i-1]) * np.linalg.norm(avg_tgt_vec)))
            results.append((src_sents[curr_i-1], combined_tgt, score, "1_2"))
            curr_i -= 1; curr_j -= 2
            
        elif step == 3: # 2-1
            combined_src = src_sents[curr_i-2] + " " + src_sents[curr_i-1]
            avg_src_vec = (src_emb[curr_i-1] + src_emb[curr_i-2]) / 2
            score = float(np.dot(avg_src_vec, tgt_emb[curr_j-1]) / (np.linalg.norm(avg_src_vec) * np.linalg.norm(tgt_emb[curr_j-1])))
            results.append((combined_src, tgt_sents[curr_j-1], score, "2_1"))
            curr_i -= 2; curr_j -= 1
            
        elif step == 4: # Skip Source
            results.append((src_sents[curr_i-1], "", 0.0, "Skip-Source"))
            curr_i -= 1
            
        elif step == 5: # Skip Target
            results.append(("", tgt_sents[curr_j-1], 0.0, "Skip-Target"))
            curr_j -= 1
        else: break
            
    return results[::-1]

def process_files():
    if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)
    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith('.csv')]
    
    for filename in files:
        print(f"\n{'='*40}")
        print(f"Đang xử lý: {filename}")
        input_path = os.path.join(INPUT_FOLDER, filename)
        
        base_name = os.path.splitext(filename)[0]
        output_clean_path = os.path.join(OUTPUT_FOLDER, f"{base_name}.csv") 
        output_score_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_score.csv")
        
        df = pd.read_csv(input_path, encoding='utf-8-sig', dtype=str, keep_default_na=False)
        df['src_id'] = pd.to_numeric(df['src_id'], errors='coerce')
        
        # Pre-process
        df = df.dropna(subset=['src_id']) 
        df['src_lang'] = df['src_lang'].fillna("").astype(str)
        df['tgt_lang'] = df['tgt_lang'].fillna("").astype(str)
        
        grouped = df.groupby('src_id')
        
        rows_score = []
        clean_aligned = []
        clean_untranslated = []
        
        # --- BIẾN THỐNG KÊ ---
        stats_counter = Counter()
        
        for src_id, group in tqdm(grouped, desc=f"Aligning {filename}", unit="doc"):
            
            src_sents_raw = [s.strip() for s in group['src_lang'].tolist() if s.strip()]
            tgt_sents_raw = [s.strip() for s in group['tgt_lang'].tolist() if s.strip()]
            
            if not src_sents_raw and not tgt_sents_raw: continue

            try:
                # Trường hợp ít câu: 1-1
                if len(src_sents_raw) <= 1 and len(tgt_sents_raw) <= 1:
                    src_text = src_sents_raw[0] if src_sents_raw else ""
                    tgt_text = tgt_sents_raw[0] if tgt_sents_raw else ""
                    
                    score = 0.0
                    if src_text and tgt_text:
                        e1 = model.encode(src_text, convert_to_numpy=True, show_progress_bar=False)
                        e2 = model.encode(tgt_text, convert_to_numpy=True, show_progress_bar=False)
                        score = float(np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2)))
                    
                    # Logic xếp loại cho trường hợp đơn giản này
                    align_type = "1_1" if (src_text and tgt_text) else ("Skip-Source" if src_text else "Skip-Target")
                    
                    rows_score.append({'src_id': src_id, 'src_lang': src_text, 'tgt_lang': tgt_text, 'score': score, 'type': align_type})
                    
                    if score >= MIN_THRESHOLD and align_type == "1_1":
                        clean_aligned.append({'src_id': src_id, 'src_lang': src_text, 'tgt_lang': tgt_text})
                        stats_counter['1_1'] += 1
                    else:
                         if src_text: 
                             clean_untranslated.append({'src_id': src_id, 'src_lang': src_text, 'tgt_lang': "Untranslated"})
                             stats_counter['Untranslated'] += 1
                    continue

                # Trường hợp nhiều câu: Dùng DP
                src_emb = get_embeddings(src_sents_raw)
                tgt_emb = get_embeddings(tgt_sents_raw)
                
                # Hàm DP giờ trả về 4 giá trị: src, tgt, score, TYPE
                aligned_data = dp_alignment_with_score(src_sents_raw, tgt_sents_raw, src_emb, tgt_emb)
                
                del src_emb, tgt_emb
                
                for s, t, score, align_type in aligned_data:
                    final_score = round(score, 4)
                    rows_score.append({'src_id': src_id, 'src_lang': s, 'tgt_lang': t, 'score': final_score, 'type': align_type})
                    
                    if s.strip():
                        if final_score >= MIN_THRESHOLD and align_type in ['1_1', '1_2', '2_1']:
                            clean_aligned.append({'src_id': src_id, 'src_lang': s, 'tgt_lang': t})
                            stats_counter[align_type] += 1
                        else:
                            clean_untranslated.append({'src_id': src_id, 'src_lang': s, 'tgt_lang': "Untranslated"})
                            stats_counter['Untranslated'] += 1
                            
            except Exception as e:
                # print(f"Error doc {src_id}: {e}")
                for s in src_sents_raw:
                    clean_untranslated.append({'src_id': src_id, 'src_lang': s, 'tgt_lang': "Untranslated"})
                    stats_counter['Untranslated'] += 1

            gc.collect()

        # --- XUẤT FILE ---
        df_aligned = pd.DataFrame(clean_aligned)
        if not df_aligned.empty: df_aligned = df_aligned.sort_values(by='src_id')
            
        df_untranslated = pd.DataFrame(clean_untranslated)
        if not df_untranslated.empty: df_untranslated = df_untranslated.sort_values(by='src_id')
        
        df_clean_final = pd.concat([df_aligned, df_untranslated], ignore_index=True)
        df_clean_final = df_clean_final[['src_id', 'src_lang', 'tgt_lang']]
        
        df_clean_final.to_csv(output_clean_path, index=False, encoding='utf-8-sig')
        
        # File score có thêm cột 'type' để bạn dễ kiểm tra
        df_score = pd.DataFrame(rows_score)[['src_id', 'src_lang', 'tgt_lang', 'score', 'type']]
        df_score.to_csv(output_score_path, index=False, encoding='utf-8-sig')
        
        # --- IN BÁO CÁO THỐNG KÊ ---
        print(f"\n--- BÁO CÁO THỐNG KÊ: {base_name}.csv ---")
        print(f"1. Tổng số cặp câu 1-1 (1_1): {stats_counter['1_1']}")
        print(f"2. Tổng số cặp câu 1-2 (1_2): {stats_counter['1_2']}")
        print(f"3. Tổng số cặp câu 2-1 (2_1): {stats_counter['2_1']}")
        print(f"4. Tổng số câu KHÔNG dịch được (Untranslated): {stats_counter['Untranslated']}")
        print(f"5. Tổng số câu output sạch: {stats_counter['1_1'] + stats_counter['1_2'] + stats_counter['2_1']}")
        print(f"--------------------------------------------\n")

        del df, df_clean_final, df_score, rows_score, clean_aligned, clean_untranslated, grouped
        gc.collect()

if __name__ == "__main__":
    process_files()

Đang chạy trên thiết bị: cpu
Đang tải model LaBSE...

Đang xử lý: JSON2_multimed_18504_17369_18504.csv


Aligning JSON2_multimed_18504_17369_18504.csv: 100%|██████████████████████████████| 1136/1136 [05:14<00:00,  3.61doc/s]



--- BÁO CÁO THỐNG KÊ: JSON2_multimed_18504_17369_18504.csv ---
1. Tổng số cặp câu 1-1 (1_1): 2456
2. Tổng số cặp câu 1-2 (1_2): 0
3. Tổng số cặp câu 2-1 (2_1): 0
4. Tổng số câu KHÔNG dịch được (Untranslated): 3
5. Tổng số câu output sạch: 2456
--------------------------------------------


Đang xử lý: PDF2_1001 Buc Thu Viet Cho Tuong Lai - Nhieu Tac Gia_876_1001.csv


Aligning PDF2_1001 Buc Thu Viet Cho Tuong Lai - Nhieu Tac Gia_876_1001.csv: 100%|███| 126/126 [00:27<00:00,  4.55doc/s]



--- BÁO CÁO THỐNG KÊ: PDF2_1001 Buc Thu Viet Cho Tuong Lai - Nhieu Tac Gia_876_1001.csv ---
1. Tổng số cặp câu 1-1 (1_1): 99
2. Tổng số cặp câu 1-2 (1_2): 3
3. Tổng số cặp câu 2-1 (2_1): 22
4. Tổng số câu KHÔNG dịch được (Untranslated): 10
5. Tổng số câu output sạch: 124
--------------------------------------------


Đang xử lý: PDF3_vui học tiếng trung qua truyện cười_76_100.csv


Aligning PDF3_vui học tiếng trung qua truyện cười_76_100.csv: 100%|███████████████████| 25/25 [00:12<00:00,  2.00doc/s]



--- BÁO CÁO THỐNG KÊ: PDF3_vui học tiếng trung qua truyện cười_76_100.csv ---
1. Tổng số cặp câu 1-1 (1_1): 84
2. Tổng số cặp câu 1-2 (1_2): 11
3. Tổng số cặp câu 2-1 (2_1): 8
4. Tổng số câu KHÔNG dịch được (Untranslated): 8
5. Tổng số câu output sạch: 103
--------------------------------------------


Đang xử lý: wikihow_vi_zh_3184_3396.csv


Aligning wikihow_vi_zh_3184_3396.csv: 100%|█████████████████████████████████████████| 213/213 [36:43<00:00, 10.34s/doc]



--- BÁO CÁO THỐNG KÊ: wikihow_vi_zh_3184_3396.csv ---
1. Tổng số cặp câu 1-1 (1_1): 21189
2. Tổng số cặp câu 1-2 (1_2): 3021
3. Tổng số cặp câu 2-1 (2_1): 3281
4. Tổng số câu KHÔNG dịch được (Untranslated): 5870
5. Tổng số câu output sạch: 27491
--------------------------------------------

